# B2-021 — Cross-modal Transformers and Advanced Vision

*30 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260901`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F4-matrices`, `book1:C7-pytorch`, `book1:C10-cnn-transfer`, `book1:C8-neural-training`, `B2-019-attention-transformers`, `B2-020-language-transformers`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../book1/units/01-F1-scientific-python/lesson.ipynb), [book1:F4-matrices](../../../book1/units/04-F4-matrices/lesson.ipynb), [book1:C7-pytorch](../../../book1/units/13-C7-pytorch/lesson.ipynb), [book1:C10-cnn-transfer](../../../book1/units/17-C10-cnn-transfer/lesson.ipynb), [book1:C8-neural-training](../../../book1/units/15-C8-neural-training/lesson.ipynb), [B2-019-attention-transformers](../B2-019-attention-transformers/lesson.ipynb), [B2-020-language-transformers](../B2-020-language-transformers/lesson.ipynb).

**Data boundary:** CPU only; use literal unit data and standard library/NumPy/Torch only. Do not use the web, downloads, external datasets, model hubs, pretrained models, checkpoints, or student data.

## Learning goals

- Convert `(B,C,H,W)` images into patch-token sequences and trace a class token to logits.
- Derive and implement box IoU, grid targets, thresholding, and class-aware NMS.
- Trace a one-down/one-up U-Net with aligned skips, pixel loss, Dice, and IoU.
- Aggregate graph neighbors and assign image/graph query, key, and value ownership.
- Train four tiny fixed-seed models without crossing immutable train/held-out splits.
- Choose classification, detection, segmentation, or graph framing and audit leakage.

## Study route

0. [Book 1/B2 bridge diagnostic](./lessons/00-book1-bridge.ipynb) — 30 minutes.
1. [Image patches and vision Transformers](./lessons/01-image-patches-and-vision-transformers.ipynb) — 90 minutes.
2. [Detection grids and set prediction](./lessons/02-detection-grids-and-set-prediction.ipynb) — 90 minutes.
3. [U-Net segmentation and skip connections](./lessons/03-unet-segmentation-and-skip-connections.ipynb) — 90 minutes.
4. [Graphs and cross-modal attention](./lessons/04-graphs-cross-modal-attention.ipynb) — 90 minutes.
5. [Vision-system design and audit](./lessons/05-vision-system-design-and-audit.ipynb) — 90 minutes.
6. [Review and self-quiz](./review.ipynb) — 60 minutes.

Complete each practice only after its `after_session` lesson. The fixture uses immutable task-specific train and held-out IDs. A repeated class label is normal; a repeated structured-feature fingerprint is not.

## Reproducibility contract

The sole seed is `20260901`. Training practices use CPU float32 features, int64 class or mask targets, stored ID order, mean loss reduction, no shuffle, dropout, scheduler, clipping, accumulation, downloads, or pretrained state. Initial state is regenerated from source; trained state and final metrics are never fixture data.


<!-- api-card -->
## Code in this unit

A quick reference for the library calls in this unit's code. Each one is explained in a comment (or in the text beside the code) where it first appears.

**New in this unit:**

- `.permute` — reorders a tensor's axes in the given order
- `torch.floor` — rounds each entry down to a whole number
- `torch.maximum` — the larger of two tensors, entry by entry
- `torch.minimum` — the smaller of two tensors, entry by entry
- `.prod` — the product of the entries (along `dim`)
- `.clamp_min` — raises every entry below a floor up to it, e.g. negatives to 0
- `torch.where` — `torch.where(cond)`: the indices where `cond` is True
- `nn.ConvTranspose2d` — learned upsampling: a transposed convolution
- `F.relu` — `max(t, 0)` for each entry
- `F.max_pool2d` — the maximum over each window, e.g. `2 × 2`, halving height and width
- `torch.bmm` — batched matrix product: `(B, n, m) @ (B, m, p)` gives `(B, n, p)`
- `.softmax` — softmax along `dim`: exponentiate, then scale to sum to 1
- `.isdisjoint` — True when two sets share no element
- `.fill_` — sets every entry to one value, in place
- `.expand` — a view that repeats length-1 axes to a bigger shape, without copying
- `math.floor` — the largest whole number `≤ x`
- `torch.all` — True when every entry is True
- `FileNotFoundError` — the built-in error for a missing file
- `.bool` — the tensor as booleans: nonzero is True, zero is False
- `F.binary_cross_entropy_with_logits` — binary cross-entropy computed from logits, with the sigmoid built in
- `torch.sigmoid` — `1 / (1 + exp(-t))` for each entry, values in `(0, 1)`
- `F.smooth_l1_loss` — squared error for small errors, absolute error for large ones

**Also used here (met earlier):**

- `nn.Linear` — affine layer `xWᵀ + b` (first explained in Book 1 C8)